# NB3 — A vs B：影像特徵有沒有增量

規格：`taiwan-housets-mvp-plan_3.md` 計畫 §5.3、§1。**這本 notebook 就是專案要的答案。**

## 假設與成功標準（計畫 §1，逐字）

> 假設：Sentinel-2 遙測指數（NDVI、NDBI）加入 XGBoost 後，房價預測 MAPE 可降低。

1. 六都測試集整體 MAPE 改善 **≥ 2 個絕對百分點**
2. 至少 **3 個都市區**達到統計顯著改善（paired t-test, p < 0.05；
   pairing 單位為「分析單元 × 季」，**非 × 月**）
3. SHAP 分析中，遙測特徵至少有 **1 項**進入前 10 重要特徵

## 這本 notebook 最重要的設計決定：**先假設答案是「沒有增量」**

一個「MAPE 改善了 2.3 個百分點」的數字，有太多種方式可以是假的：

| 假陽性來源 | 這裡怎麼擋 |
|---|---|
| 單一次 train/test 的隨機性 | **多個 seed**（`N_SEEDS`），報分布不報單一數字 |
| 「多了 5 個特徵」本身就會讓樹模型變好 | **負對照 N1**：把影像特徵換成同維度的**隨機雜訊** |
| 影像特徵其實只是單元的固定效果（不是時間訊號） | **負對照 N2**：在單元內把影像特徵**時間打散** |
| 季度影像 × 月度房價的結構性重複壓低 p-value | pairing 用「單元 × 季」；**N3** 直接用季粒度重跑 |
| 被省略的交通特徵讓影像代它領功 | **N4**：排除窗口內有新站通車的單元（NB2 已記錄這個偏誤） |
| 所得遞延造成的偽訊號 | **N5**：排除所得被遞延的年份 |

**負對照 N1／N2 是這裡最關鍵的兩項。** 如果隨機雜訊也能讓 MAPE 改善 2 個百分點，
那第 1 條成功標準本身就沒有意義；如果打散後改善還在，那影像帶的是空間固定效果，
不是計畫假設的「都市變遷訊號」。

---
## 0. 環境與上游產出

In [ ]:
# Colab 執行時取消註解
# !pip install -q xgboost lightgbm scikit-learn shap pyarrow

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 190)
pd.set_option("display.max_columns", 80)

try:
    import google.colab  # noqa: F401
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/TW_Housets")
except ImportError:
    _cwd = Path.cwd()
    PROJECT = _cwd.parent if _cwd.name == "notebooks" else _cwd

PROC = PROJECT / "data" / "processed"
NEED = {"NB2 大表": PROC / "modeling_table.parquet",
        "NB2 中介資料": PROC / "merge_meta.json",
        "NB1 中介資料": PROC / "rs_meta.json"}
_miss = {k: v for k, v in NEED.items() if not v.exists()}
if _miss:
    raise FileNotFoundError("缺少上游產出：\n"
                            + "\n".join(f"  {k}：{v}" for k, v in _miss.items()))

MM = json.loads(NEED["NB2 中介資料"].read_text(encoding="utf-8"))
# NB1 的中介資料：需要它的 I3「solar day 不足的季」來決定訓練起點——
# 差分特徵的**基期**若落在不可用的季，那一整年的 B 訊號都是雜訊。
RSM = json.loads(NEED["NB1 中介資料"].read_text(encoding="utf-8"))
if not MM["checks"]["passed"]:
    print("⚠️  NB2 的合理性檢查沒有全部通過——先看 merge_meta.json 的 checks 再繼續。")
print(f"粒度：{MM['scheme']}｜結構性重複：{MM['structural_repeat_with_price']}")
print("NB2 記錄的已知偏誤：")
for b in MM["known_biases"]:
    print(f"  · {b}")

tbl = pd.read_parquet(NEED["NB2 大表"])
tbl["period"] = pd.PeriodIndex(tbl["period"], freq="M" if MM["time_unit"] == "月" else "Q")
tbl["quarter"] = pd.PeriodIndex(tbl["quarter"], freq="Q")
tbl = tbl[tbl["keep"]].copy()
print(f"\n大表：{len(tbl):,} 列（單元 {tbl.unit.nunique()}、期 {tbl.period.nunique()}）")

TIERS = {k: [c for c in v if c in tbl.columns]
         for k, v in MM["feature_tiers"].items() if k != "img"}
IMG_FEATS = [c for c in MM["feature_tiers"]["img"] if c in tbl.columns]
PRIMARY_TIER = "A1"          # 計畫 §5.3 的字面定義（含建物屬性）
FEATS_A = TIERS[PRIMARY_TIER]
FEATS_B = FEATS_A + IMG_FEATS
TARGET = MM["target"]
for k, v in TIERS.items():
    print(f"特徵層 {k}：{len(v)} 個")
print(f"影像特徵：{IMG_FEATS}｜主層 = {PRIMARY_TIER}")
# 100% NaN 的特徵要拿掉：它沒有資訊，但會改變 colsample 的 RNG 路徑，
# 實測光是移除一個零資訊欄就能讓 ΔMAPE 動 0.7 pp（純雜訊）。
_allnan = [c for c in set(sum(TIERS.values(), [])) | set(IMG_FEATS)
           if tbl[c].isna().all()]
if _allnan:
    print(f"⚠️  移除 {len(_allnan)} 個全 NaN 的特徵：{_allnan}")
    TIERS = {k: [c for c in v if c not in _allnan] for k, v in TIERS.items()}
    IMG_FEATS = [c for c in IMG_FEATS if c not in _allnan]
    FEATS_A = TIERS[PRIMARY_TIER]
    FEATS_B = FEATS_A + IMG_FEATS

---
## 1. 時間切分

計畫 §5.3 寫 train 2017Q1–2022Q4。**那個起點用不了**——NB1 實測 2017 年台灣沒有
可用的 Sentinel-2（2017Q2 連一張雲量 < 40% 的都沒有），而年際差分還要再往後一年。
所以起點改成 **2019Q1**，並且**下限由資料自己決定**：取「影像差分特徵齊全」的最早一季。

In [ ]:
# 起點要同時滿足兩件事，**不能只用 min()**：
#   (a) 差分特徵在**絕大多數**單元上齊全（用 min() 的話一個單元就能把起點拉早）
#   (b) 差分的**基期**（Q−4）本身是可用的季——NB1 已經判定 2017 各季 solar day
#       過少而不可用，而 dNDVI(2018Q1) 的基期正是 2017Q1（實測 solar day 中位 2）
_dcols = [c for c in IMG_FEATS if c.startswith("d")]
_cov = (tbl.assign(_ok=tbl[_dcols].notna().all(axis=1))
           .groupby("quarter")["_ok"].mean())
DIFF_COVER_MIN = 0.90
_cand = [q for q, v in _cov.items() if v >= DIFF_COVER_MIN]
# NB1 記錄的不可用季（solar day 不足）→ 基期落在裡面的季一律排除
_thin = set(pd.PeriodIndex(RSM["checks"].get("I3", {}).get("thin_quarters", []), freq="Q")) \
    if RSM["checks"].get("I3", {}).get("thin_quarters") else set()
_cand = [q for q in _cand if (q - 4) not in _thin]
TRAIN_START = min(_cand) if _cand else tbl["quarter"].min()
print(f"差分覆蓋率 ≥ {DIFF_COVER_MIN:.0%} 且基期可用的最早一季：{TRAIN_START}")
print(f"  NB1 判定 solar day 不足的季：{sorted(str(q) for q in _thin) or '（無）'}")
print(f"  逐季差分覆蓋率（前 8 季）：{_cov.head(8).round(2).to_dict()}")
TRAIN_END, VAL_END, TEST_END = (pd.Period("2022Q4", "Q"), pd.Period("2023Q2", "Q"),
                                pd.Period("2024Q4", "Q"))
print(f"差分特徵齊全的最早一季：{TRAIN_START}（計畫寫 2017Q1，實際用不了）")

def split_of(q):
    if q < TRAIN_START:
        return "unusable"
    if q <= TRAIN_END:
        return "train"
    if q <= VAL_END:
        return "val"
    if q <= TEST_END:
        return "test"
    return "future"

tbl["split"] = tbl["quarter"].map(split_of)
print(tbl.groupby("split").agg(列數=("unit", "size"), 單元=("unit", "nunique"),
                               季=("quarter", "nunique")).to_string())
data = tbl[tbl["split"].isin(["train", "val", "test"])].copy()
# 影像差分在最早一年是 NaN；樹模型吃得下 NaN，但要確認不是整欄都缺
_nan = data[IMG_FEATS].isna().mean()
print(f"\n影像特徵的 NaN 佔比：{_nan.round(3).to_dict()}")
if (_nan > 0.5).any():
    raise RuntimeError(f"有影像特徵超過一半是 NaN：{_nan[_nan > 0.5].to_dict()}——"
                       "切分起點或 NB1 的抽取有問題。")

---
## 2. 模型設定

**A 與 B 用完全相同的超參數與相同的 seed。** 不對 B 額外調參——否則比較的是
「調參的成果」而不是「影像特徵的增量」。多個 seed 是為了量出 run 之間的變異：
一個 2 個百分點的差距如果小於 seed 之間的標準差，那它就不是結果。

In [ ]:
import xgboost as xgb

N_SEEDS = 20          # 實測 ΔMAPE 對 seed 批次的敏感度約 ±0.4 pp，10 個不夠
# **容量要對齊。** B 比 A 多 5 個特徵，`colsample_bytree=0.8` 會讓 B 每棵樹抽到
# 更多欄（0.8×34=28 vs 0.8×39=32），而 early stopping 又讓 B 平均多訓 109 棵樹
# （實測 A 192 棵 vs B 301 棵）。實測把每棵樹的欄數固定成相同絕對值之後，
# ΔMAPE 動 0.6 pp——那個量級足以決定「≥ 2 pp」過不過。
# 做法：colsample_bytree = COLSAMPLE_K / 特徵數，兩邊每棵樹都抽同樣多欄。
COLSAMPLE_K = 27
PARAMS = dict(n_estimators=600, max_depth=6, learning_rate=0.05,
              subsample=0.8,
              min_child_weight=5, reg_lambda=1.0,
              objective="reg:squarederror", tree_method="hist")
EARLY_STOP = 50
LOG_TARGET = True


def mape(y, p):
    return float(np.mean(np.abs((y - p) / y)) * 100)


def mdape(y, p):
    return float(np.median(np.abs((y - p) / y)) * 100)


def fit_predict(feats, seed: int, d=None, log_target: bool = None):
    """回傳 (test 的預測, 模型)。log 目標讓誤差在比例上對稱（房價右偏）。

    注意 log 反推有 Jensen 偏誤（實測平均相對偏誤 −8.7%），所以主結果同時報 MdAPE。
    """
    log_target = LOG_TARGET if log_target is None else log_target
    d = data if d is None else d
    tr, va, te = (d[d.split == s] for s in ("train", "val", "test"))
    ytr = np.log(tr[TARGET]) if log_target else tr[TARGET]
    yva = np.log(va[TARGET]) if log_target else va[TARGET]
    m = xgb.XGBRegressor(**PARAMS, random_state=seed,
                         colsample_bytree=min(1.0, COLSAMPLE_K / max(len(feats), 1)),
                         early_stopping_rounds=EARLY_STOP)
    m.fit(tr[feats], ytr, eval_set=[(va[feats], yva)], verbose=False)
    p = m.predict(te[feats])
    return (np.exp(p) if log_target else p), m


def run_ab(feats_a, feats_b, d=None, n_seeds=N_SEEDS, tag=""):
    """同一組 seed 跑 A 與 B，回傳逐 seed 的指標與逐列誤差。"""
    d = data if d is None else d
    te = d[d.split == "test"]
    rows, errs = [], []
    for s in range(n_seeds):
        pa, _ = fit_predict(feats_a, s, d)
        pb, _ = fit_predict(feats_b, s, d)
        y = te[TARGET].to_numpy()
        rows.append({"seed": s, "MAPE_A": mape(y, pa), "MAPE_B": mape(y, pb),
                     "MdAPE_A": mdape(y, pa), "MdAPE_B": mdape(y, pb)})
        errs.append(pd.DataFrame({
            "unit": te["unit"].to_numpy(), "縣市": te["縣市"].to_numpy(),
            "quarter": te["quarter"].astype(str).to_numpy(), "seed": s,
            "ape_A": np.abs((y - pa) / y) * 100, "ape_B": np.abs((y - pb) / y) * 100}))
    r = pd.DataFrame(rows)
    r["ΔMAPE"] = r["MAPE_A"] - r["MAPE_B"]          # 正值 = B 比較好
    r["ΔMdAPE"] = r["MdAPE_A"] - r["MdAPE_B"]
    if tag:
        print(f"[{tag}] MAPE_A {r.MAPE_A.mean():.2f}±{r.MAPE_A.std():.2f}"
              f"｜MAPE_B {r.MAPE_B.mean():.2f}±{r.MAPE_B.std():.2f}"
              f"｜ΔMAPE {r['ΔMAPE'].mean():+.2f}±{r['ΔMAPE'].std():.2f}"
              f"｜ΔMdAPE {r['ΔMdAPE'].mean():+.2f} pp")
    return r, pd.concat(errs, ignore_index=True)

---
## 3. 主結果：A vs B

In [ ]:
# **雙口徑。** `keep=True` 同時要求「影像可用」，也就是評估集被影像品質挑過一遍
# ——實測那讓 B 佔約 0.21 pp 的便宜。所以同時報「只要求 n_tx ≥ 3、讓 B 在影像缺失
# 的 cell 上吃 NaN」這個較不利於 B 的口徑，兩者的差就是 selection 的敏感度。
res, errs = run_ab(FEATS_A, FEATS_B, tag="主結果（keep：n_tx≥3 且影像可用）")
print()
print(res.round(3).to_string(index=False))
print(f"\n測試集：{int((data.split == 'test').sum()):,} 列"
      f"（{data[data.split == 'test'].unit.nunique()} 單元 × "
      f"{data[data.split == 'test'].quarter.nunique()} 季）")

dm, ds = res["ΔMAPE"].mean(), res["ΔMAPE"].std()
print(f"\nΔMAPE（A − B，正值代表影像有幫助）：{dm:+.3f} pp，"
      f"seed 間標準差 {ds:.3f} pp")
if abs(dm) < ds:
    print("  ⚠️  平均差距小於 seed 間的標準差——這個差距**不能當成結果**。")
crit1 = bool(dm >= 2.0)
print(f"成功標準 1（ΔMAPE ≥ 2 pp）：{'✅ 達成' if crit1 else '❌ 未達成'}")

In [ ]:
# ---- 逐縣市 ----
by_city = (errs.groupby(["縣市", "seed"])[["ape_A", "ape_B"]].mean()
               .groupby("縣市").mean())
by_city["ΔMAPE"] = by_city["ape_A"] - by_city["ape_B"]
CITY_ORDER = ["臺北市", "新北市", "桃園市", "臺中市", "臺南市", "高雄市"]
print("逐縣市 MAPE（跨 seed 平均）")
print(by_city.reindex([c for c in CITY_ORDER if c in by_city.index])
             .round(2).to_string())

---
## 4. 統計檢定：pairing 單位是「單元 × 季」

計畫 §5.3 說得很清楚：若房價是月粒度，同季 3 個月共用同一組遙測值，
以「單元 × 月」為 pair 會讓 3 個 pair 不獨立、**p-value 被人為壓低**。
所以先把同季各月的誤差平均，再做 paired t-test。

下面同時報「× 月」的 p-value 作為對照——不是要用它，是要**看出它有多樂觀**。

In [ ]:
def paired_test(e: pd.DataFrame, by: list) -> dict:
    """先在 by 這個粒度上把誤差平均（跨 seed 也平均），再做 paired t-test。"""
    g = e.groupby(by)[["ape_A", "ape_B"]].mean()
    d = (g["ape_A"] - g["ape_B"]).dropna()
    if len(d) < 3:
        return {"n_pair": len(d), "mean_diff": np.nan, "t": np.nan, "p": np.nan,
                "wilcoxon_p": np.nan}
    t, p = stats.ttest_rel(g["ape_A"], g["ape_B"])
    w = stats.wilcoxon(g["ape_A"], g["ape_B"])[1] if len(d) >= 10 else np.nan
    return {"n_pair": int(len(d)), "mean_diff": float(d.mean()),
            "t": float(t), "p": float(p), "wilcoxon_p": float(w)}


def cluster_test(e: pd.DataFrame, n_boot: int = 4000, seed: int = 0) -> dict:
    """**以單元為 cluster** 的檢定。這是主要的檢定，上面那個只當對照。

    為什麼必須 cluster：以 (單元 × 季) 當獨立樣本會高估樣本數——同一單元不同季的
    ΔAPE 高度相關（實測 ICC ≈ 0.56，188 個名目 pair 的有效樣本只有約 104）。
    計畫 §5.3 只點名了「× 月」這個假複製，但「× 季」還有第二層假複製在單元上。
    實測後果是決策性的：逐縣市的 p 值從 0.004 變成 0.073。
    """
    g = (e.groupby(["unit", "quarter"])[["ape_A", "ape_B"]].mean()
          .assign(diff=lambda x: x["ape_A"] - x["ape_B"]).reset_index())
    per_unit = g.groupby("unit")["diff"].mean().dropna()
    if len(per_unit) < 3:
        return {"n_cluster": len(per_unit), "mean_diff": np.nan, "p": np.nan,
                "ci_lo": np.nan, "ci_hi": np.nan, "icc": np.nan}
    t, p = stats.ttest_1samp(per_unit, 0.0)
    # cluster bootstrap（重抽單元，不重抽季）
    rng = np.random.default_rng(seed)
    units = per_unit.index.to_numpy()
    boot = np.array([per_unit.loc[rng.choice(units, len(units), replace=True)].mean()
                     for _ in range(n_boot)])
    # ICC：單元間變異 / 總變異
    gm = g["diff"].mean()
    ss_b = ((g.groupby("unit")["diff"].transform("mean") - gm) ** 2).sum()
    ss_t = ((g["diff"] - gm) ** 2).sum()
    return {"n_cluster": int(len(per_unit)), "n_cell": int(len(g)),
            "mean_diff": float(per_unit.mean()), "t": float(t), "p": float(p),
            "boot_p": float(2 * min((boot <= 0).mean(), (boot >= 0).mean())),
            "ci_lo": float(np.percentile(boot, 2.5)),
            "ci_hi": float(np.percentile(boot, 97.5)),
            "icc": float(ss_b / ss_t) if ss_t > 0 else np.nan}


errs["period"] = errs.get("period", errs["quarter"])
overall_cl = cluster_test(errs)
overall_q = paired_test(errs, ["unit", "quarter"])
overall_mo = paired_test(errs, ["unit", "period_m"]) if "period_m" in errs else None
overall_sd = paired_test(errs, ["unit", "quarter", "seed"])
print("整體檢定——**由嚴到鬆**，用最嚴的那個下判斷")
print(f"  ① 單元 cluster（主要）：n_cluster={overall_cl['n_cluster']}"
      f"（{overall_cl['n_cell']} 個 cell）, mean {overall_cl['mean_diff']:+.3f} pp, "
      f"p={overall_cl['p']:.4g}, bootstrap p={overall_cl['boot_p']:.4g}, "
      f"95% CI [{overall_cl['ci_lo']:+.3f}, {overall_cl['ci_hi']:+.3f}], "
      f"ICC={overall_cl['icc']:.3f}")
print(f"  ② 單元 × 季（計畫指定，但同單元跨季相關）：n={overall_q['n_pair']}, "
      f"p={overall_q['p']:.4g}")
print(f"  ③ 單元 × 季 × seed（**錯的**，seed 不獨立）：n={overall_sd['n_pair']}, "
      f"p={overall_sd['p']:.4g}")
print(f"  ↑ ICC {overall_cl['icc']:.2f} 代表 ② 的名目 n 高估了有效樣本數。"
      "計畫 §5.3 只點名「× 月」這一層假複製，單元上還有第二層。")

rows = []
for city, g in errs.groupby("縣市"):
    r = cluster_test(g, n_boot=2000)
    r["縣市"] = city
    r["p_naive_cell"] = paired_test(g, ["unit", "quarter"])["p"]
    r["顯著改善"] = bool(r["p"] < 0.05 and r["mean_diff"] > 0)
    rows.append(r)
city_test = pd.DataFrame(rows).set_index("縣市").reindex(
    [c for c in CITY_ORDER if c in {r["縣市"] for r in rows}])
print("\n逐縣市（p 為單元 cluster；p_naive_cell 為以 cell 當獨立樣本的對照）")
print(city_test.round(4).to_string())
_missing_city = [c for c in CITY_ORDER if c not in city_test.index]
if _missing_city:
    print(f"\n⚠️  測試集裡完全沒有這些縣市：{_missing_city}"
          "——「≥ 3 個縣市」的分母因此不是 6，結論要標註。")
n_sig = int(city_test["顯著改善"].sum())
crit2 = bool(n_sig >= 3)
print(f"\n成功標準 2（≥ 3 個縣市顯著改善，單元 cluster）：{n_sig} 個 → "
      f"{'✅ 達成' if crit2 else '❌ 未達成'}")

---
## 5. SHAP：只解讀符號與排名

計畫 §5.3 說明得很清楚：季度影像 × 月度房價的結構性重複會讓 SHAP 的**量級**偏誤，
而偏誤方向不確定（樣本重複加重分裂權重 → 偏高；季內無變異 → 偏低）。
所以**只看影像特徵有沒有進前 10、以及方向**，不解讀數值大小。

In [ ]:
import shap

_, model_b = fit_predict(FEATS_B, 0)
te = data[data.split == "test"]
expl = shap.TreeExplainer(model_b)
sv = expl.shap_values(te[FEATS_B])
imp = (pd.DataFrame({"feature": FEATS_B,
                     "mean_abs_shap": np.abs(sv).mean(axis=0)})
       .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
imp["rank"] = imp.index + 1
imp["是影像特徵"] = imp["feature"].isin(IMG_FEATS)
print("SHAP 重要性排名（前 15）")
print(imp.head(15).to_string(index=False))

img_ranks = imp.loc[imp["是影像特徵"], ["feature", "rank"]]
crit3 = bool((img_ranks["rank"] <= 10).any())
print(f"\n影像特徵的排名：{img_ranks.set_index('feature')['rank'].to_dict()}")
print(f"成功標準 3（≥ 1 項影像特徵進前 10）：{'✅ 達成' if crit3 else '❌ 未達成'}")

# 方向：SHAP 與特徵值的相關符號
sign = {}
for f in IMG_FEATS:
    j = FEATS_B.index(f)
    x = te[f].to_numpy()
    ok = np.isfinite(x)
    sign[f] = (round(float(np.corrcoef(x[ok], sv[ok, j])[0, 1]), 3)
               if ok.sum() > 10 else None)
print(f"影像特徵的作用方向（SHAP 與特徵值的相關）：{sign}")

---
## 6. 負對照與 robustness

**這一節比第 3 節重要。** 第 3 節只告訴我們「有沒有差」，這一節告訴我們
「那個差是不是影像帶來的」。

In [ ]:
checks = {}

# ---- N1：null 影像對照（主對照 = 單元輪換）----
# 目的是問「同樣多、同樣長相的特徵，但**與這個單元無關**，能買到多少 MAPE？」
#
# 標準常態雜訊是不夠格的對照：它沒有時間自相關、沒有空間自相關、沒有 NaN 型態，
# 樹模型對它的反應與對真實影像不同。實測標準常態**高估**了「加欄的代價」約 0.34 pp，
# 於是 `net_d` 會系統性偏向支持影像假設。
#
# 主對照 **N1d 單元輪換**：把每個單元的影像時間序列整條換給另一個單元。
# 邊際分布、時間自相關、NaN 型態全部保留，只錯配 unit ↔ 影像。
# 若真實影像與「隔壁鄉鎮的影像」在統計上分不出來，那就沒有增量。
_units = sorted(data["unit"].unique())
_rot = {u: _units[(i + max(1, len(_units) // 3)) % len(_units)]
        for i, u in enumerate(_units)}
# 影像在同一季內是常數（NB2 的 M2 已驗證），所以先去重再建索引——
# 房價是月粒度時每個 (unit, quarter) 有 3 列，直接 set_index 會是非唯一索引。
_img_lookup = (data.drop_duplicates(subset=["unit", "quarter"])
                   .set_index(["unit", "quarter"])[IMG_FEATS])
d_rot = data.copy()
_key = pd.MultiIndex.from_arrays([d_rot["unit"].map(_rot), d_rot["quarter"]])
for f in IMG_FEATS:
    d_rot[f] = _img_lookup[f].reindex(_key).to_numpy()
print(f"N1d 單元輪換：影像值改成別的單元同季的值"
      f"（輪換後仍有值的比例 {d_rot[IMG_FEATS].notna().all(axis=1).mean():.1%}）")
res_n1, _ = run_ab(FEATS_A, FEATS_B, d=d_rot, n_seeds=N_SEEDS, tag="N1d 單元輪換（主對照）")
checks["N1_unit_rotated"] = {"dMAPE_mean": round(float(res_n1["ΔMAPE"].mean()), 3),
                             "dMAPE_sd": round(float(res_n1["ΔMAPE"].std()), 3)}

# 次要對照：標準常態雜訊。保留它只是為了量「純粹多幾欄」的代價，不當主對照。
rng = np.random.default_rng(0)
d_noise = data.copy()
noise_cols = []
for i, f in enumerate(IMG_FEATS):
    c = f"noise_{i}"
    d_noise[c] = rng.standard_normal(len(d_noise))
    noise_cols.append(c)
res_n1b, _ = run_ab(FEATS_A, FEATS_A + noise_cols, d=d_noise, n_seeds=5,
                    tag="N1b 標準常態雜訊（次要）")
checks["N1b_gaussian"] = {"dMAPE_mean": round(float(res_n1b["ΔMAPE"].mean()), 3),
                          "dMAPE_sd": round(float(res_n1b["ΔMAPE"].std()), 3)}

# **同 seed、同資料的直接對照**：把 null 影像與真實影像放在同一次比較裡，
# A 項才會抵銷掉（用兩組不同 seed 相減會多帶進 A 的變異）。
_direct = []
for sd in range(N_SEEDS):
    p_null, _ = fit_predict(FEATS_B, sd, d_rot)
    p_real, _ = fit_predict(FEATS_B, sd, data)
    _y = data[data.split == "test"][TARGET].to_numpy()
    _yn = d_rot[d_rot.split == "test"][TARGET].to_numpy()
    _direct.append(mape(_yn, p_null) - mape(_y, p_real))
net_direct = float(np.mean(_direct))
checks["net_vs_null_direct"] = {"mean": round(net_direct, 3),
                                "sd": round(float(np.std(_direct)), 3)}
print(f"\n真實影像 vs 輪換影像（同 seed 直接對照）：{net_direct:+.3f} "
      f"± {np.std(_direct):.3f} pp")
print("  ↑ **這個數字才是「影像認得出自己的單元」的證據。** 接近 0 就代表認不出來。")

In [ ]:
# ---- N2：影像特徵在單元內時間打散 ----
# 保留每個單元的影像分布（也就是保留空間固定效果），只破壞時間對應。
# 若改善還在，那影像帶的是「哪個區」而不是「這個區在變」——後者才是計畫的假設。
# **在 (單元 × split) 內打散，不跨切分邊界。** 跨邊界打散會讓測試集的影像值大量
# 取自訓練期（實測 66.5%），使打散版看到的分布比真實情況更接近訓練期、
# 人為讓對照組變好——實測 `time_d` 的符號就是由「跨不跨邊界」決定的（−0.05 vs +0.20）。
d_shuf = data.copy()
rs2 = np.random.default_rng(1)
for f in IMG_FEATS:
    d_shuf[f] = (d_shuf.groupby(["unit", "split"])[f]
                       .transform(lambda s: rs2.permutation(s.to_numpy())))
res_n2, _ = run_ab(FEATS_A, FEATS_B, d=d_shuf, n_seeds=N_SEEDS,
                   tag="N2 單元×split 內時間打散")
checks["N2_shuffled"] = {"dMAPE_mean": round(float(res_n2["ΔMAPE"].mean()), 3),
                         "dMAPE_sd": round(float(res_n2["ΔMAPE"].std()), 3),
                         "within": "unit×split（不跨邊界）"}

In [ ]:
# ---- N3：季粒度重跑（消除結構性重複）----
if MM["structural_repeat_with_price"]:
    q = (data.groupby(["unit", "縣市", "quarter", "split"], observed=True)
             .agg({TARGET: "median", **{c: "mean" for c in FEATS_B if c in data.columns}})
             .reset_index())
    q["quarter"] = pd.PeriodIndex(q["quarter"].astype(str), freq="Q")
    res_n3, errs_n3 = run_ab(FEATS_A, FEATS_B, d=q, n_seeds=5, tag="N3 季粒度")
    t_n3 = paired_test(errs_n3, ["unit", "quarter"])
    checks["N3_quarterly"] = {"dMAPE_mean": round(float(res_n3["ΔMAPE"].mean()), 3),
                              "dMAPE_sd": round(float(res_n3["ΔMAPE"].std()), 3),
                              "p": None if pd.isna(t_n3["p"]) else round(t_n3["p"], 5)}
    print(f"  季粒度的 paired t-test：n={t_n3['n_pair']}, p={t_n3['p']:.4g}")
else:
    checks["N3_quarterly"] = {"skipped": "房價本來就是季粒度"}
    print("N3 跳過：房價本來就是季粒度，沒有結構性重複")

In [ ]:
# ---- N4：排除窗口內有新捷運站的單元 ----
# NB2 記錄的偏誤：OSM 車站的 start_date 覆蓋率只有 11%，所以「通車前後」特徵做不出來。
# 這裡用**能辨識的那部分**把有新站的單元排除掉——不完整，但方向明確：
# 若影像的增益只存在於有新站的單元，那增益很可能是被省略的交通特徵。
POI_DIR = PROJECT / "data" / "raw" / "poi"
new_station_units = set()
if POI_DIR.exists():
    import geopandas as gpd
    from shapely.geometry import Point

    _bnd = next(iter(sorted((PROJECT / "data" / "raw" / "boundary").rglob("*.shp"))), None)
    _poi = pd.concat([pd.read_parquet(p) for p in sorted(POI_DIR.glob("*.parquet"))],
                     ignore_index=True) if any(POI_DIR.glob("*.parquet")) else None
    if _bnd is not None and _poi is not None:
        st = _poi[(_poi["kind"] == "station") & _poi["start_date"].notna()].copy()
        st["y"] = pd.to_numeric(st["start_date"].str.slice(0, 4), errors="coerce")
        st = st[(st["y"] >= data["quarter"].min().year) & (st["y"] <= TEST_END.year)]
        if len(st):
            g = gpd.read_file(_bnd).to_crs(4326)
            g["unit"] = g["COUNTYNAME"] + g["TOWNNAME"]
            gs = gpd.GeoDataFrame(st, geometry=[Point(xy) for xy in zip(st.lon, st.lat)],
                                  crs=4326)
            j = gpd.sjoin(gs, g[["unit", "geometry"]], how="left", predicate="within")
            new_station_units = set(j["unit"].dropna())
print(f"窗口內有新站（OSM 有 start_date 的部分）的單元：{len(new_station_units)} 個")
if new_station_units:
    d_n4 = data[~data["unit"].isin(new_station_units)]
    res_n4, _ = run_ab(FEATS_A, FEATS_B, d=d_n4, n_seeds=5, tag="N4 排除新站單元")
    checks["N4_no_new_station"] = {
        "n_units_excluded": len(new_station_units),
        "dMAPE_mean": round(float(res_n4["ΔMAPE"].mean()), 3),
        "dMAPE_sd": round(float(res_n4["ΔMAPE"].std()), 3)}
else:
    checks["N4_no_new_station"] = {"skipped": "OSM 沒有可用的 start_date"}
    print("N4 跳過：OSM 沒有可用的 start_date（這一項的偏誤因此沒有被排除，"
          "要寫在結論的限制裡）")
# **不要因為跑了 N4 就當這個偏誤解決了。** 實測 900 個站只有 132 個（14.7%）有
# start_date，窗口內約 60 站，最後只排除掉 5/102 個單元。這個偏誤**實質上仍然存在**，
# 而且方向是偏袒影像。要真的排除需要交通部／各捷運公司的通車年表（六都的通車站
# 是可窮舉的，幾十筆），那還沒做。
checks["N4_no_new_station"]["coverage_warning"] = (
    "OSM start_date 覆蓋率僅約 15%，此 robustness 只碰到少數單元，"
    "「缺少通車前後特徵而偏袒影像」這個偏誤實質上未被排除")

In [ ]:
# ---- 特徵層敏感度（A0 / A1 / A2）----
# 「A 怎麼定義」造成的 ΔMAPE 分散實測比 2 pp 的成功門檻還大，所以三層都要報，
# 而且結論只在**符號一致**時才成立。
tier_res = {}
for name, fa in TIERS.items():
    if not fa:
        continue
    rt, _ = run_ab(fa, fa + IMG_FEATS, n_seeds=5, tag=f"層 {name}（{len(fa)} 特徵）")
    tier_res[name] = {"n_feats": len(fa),
                      "dMAPE_mean": round(float(rt["ΔMAPE"].mean()), 3),
                      "dMAPE_sd": round(float(rt["ΔMAPE"].std()), 3)}
checks["feature_tiers"] = tier_res
_signs = {k: np.sign(v["dMAPE_mean"]) for k, v in tier_res.items()}
tier_consistent = len(set(_signs.values())) == 1
print(f"\n三層的 ΔMAPE 符號是否一致：{tier_consistent}（{_signs}）")
checks["tier_sign_consistent"] = bool(tier_consistent)

# ---- N5：排除所得被遞延的年份 ----
_IY = "income_source_year" if "income_source_year" in data.columns else "income_year_used"
if _IY in data.columns and (data[_IY] != data["year"] - MM.get("income_lag_years", 0)).any():
    # **只在 test 上排除**。把遞延列從所有 split 拿掉會同時砍掉訓練資料與測試窗口，
    # 那樣就分不清「排除遞延」與「test 變短」哪個造成了差異。
    _bad_test = ((data["split"] == "test")
                 & (data[_IY] != data["year"] - MM.get("income_lag_years", 0)))
    d_n5 = data[~_bad_test]
    if (d_n5["split"] == "test").sum() > 50:
        res_n5, _ = run_ab(FEATS_A, FEATS_B, d=d_n5, n_seeds=5, tag="N5 排除所得遞延年")
        checks["N5_income_actual"] = {
            "dMAPE_mean": round(float(res_n5["ΔMAPE"].mean()), 3),
            "dMAPE_sd": round(float(res_n5["ΔMAPE"].std()), 3)}
    else:
        checks["N5_income_actual"] = {"skipped": "排除後測試集太小"}
        print("N5 跳過：排除遞延年之後測試集太小")
else:
    checks["N5_income_actual"] = {"skipped": "沒有遞延"}
    print("N5 跳過：所得沒有被遞延")

---
## 7. 結論

In [ ]:
NOISE_TOL = 0.5
main_d = float(res["ΔMAPE"].mean())
main_sd = float(res["ΔMAPE"].std())
main_se = main_sd / np.sqrt(len(res))
null_d = checks["N1_unit_rotated"]["dMAPE_mean"]
shuf_d = checks["N2_shuffled"]["dMAPE_mean"]
n3 = checks.get("N3_quarterly", {})

# 影像相對於「別的單元的影像」的淨增量（同 seed 直接對照，A 項已抵銷）
net_d = checks["net_vs_null_direct"]["mean"]
time_d = main_d - shuf_d

# **null 結果與「有增量但對照失敗」是兩件不同的事，不能都叫 PARTIAL。**
is_null = bool(abs(main_d) < 2 * main_se or main_d <= 0)
controls = {}
controls["null_imagery_small"] = bool(abs(null_d) < NOISE_TOL)
controls["time_signal_positive"] = bool(time_d > NOISE_TOL)
controls["net_vs_null_positive"] = bool(net_d > NOISE_TOL)
controls["quarterly_same_sign"] = (
    None if "skipped" in n3 else bool(np.sign(n3.get("dMAPE_mean", 0)) == np.sign(main_d)))
controls["tier_sign_consistent"] = bool(checks.get("tier_sign_consistent", False))
# 主結果是 null 的時候，「負對照有沒有過」本身沒有意義——不要印成「主結果不可信」
controls_status = ("na_null_result" if is_null else
                   "ok" if all(v for v in controls.values() if v is not None) else
                   "failed")

all_met = crit1 and crit2 and crit3
n_met = int(crit1) + int(crit2) + int(crit3)
verdict = {
    "criteria": {
        "1_mape_ge_2pp": {"target": 2.0, "actual": round(main_d, 3),
                          "sd_across_seeds": round(main_sd, 3),
                          "se": round(float(main_se), 3),
                          "cluster_ci": [round(overall_cl["ci_lo"], 3),
                                         round(overall_cl["ci_hi"], 3)],
                          "met": crit1},
        "2_cities_significant": {"target": 3, "actual": n_sig,
                                 "cities_in_test": list(city_test.index),
                                 "cities_missing": _missing_city, "met": crit2},
        "3_shap_top10": {"met": crit3,
                         "img_ranks": img_ranks.set_index("feature")["rank"].to_dict()},
    },
    "n_criteria_met": n_met,
    "controls": controls,
    "controls_status": controls_status,
    "negative_controls": checks,
    "derived": {
        "net_vs_null_pp": round(net_d, 3),
        "time_signal_pp": round(time_d, 3),
        "delta_mdape_pp": round(float(res["ΔMdAPE"].mean()), 3),
        "is_null": is_null,
        "exceeds_seed_noise": bool(abs(main_d) > main_sd),
        "mean_exceeds_2se": bool(abs(main_d) > 2 * main_se),
    },
}
# **crit1（MAPE 有沒有降低）是主判準，crit3（SHAP 排名）不是它的替代品。**
# 影像特徵進得了 SHAP 前 10 只代表「樹用了它」，不代表 out-of-sample 有增量——
# 本次實測就是這一格：NDVI 排第 10，而 ΔMAPE 是 −0.58 pp。
# 所以 main_d ≤ 0 時一律 NOT_SUPPORTED，不能因為 crit3 過了就叫 PARTIAL。
_sig_negative = bool(overall_cl["ci_hi"] < 0)
verdict["derived"]["significantly_negative"] = _sig_negative
verdict["overall"] = (
    "INVALID" if all_met and controls_status == "failed" else
    "SUPPORTED" if all_met and controls_status == "ok" else
    "NOT_SUPPORTED_NEGATIVE" if _sig_negative else
    "NOT_SUPPORTED" if main_d <= 0 or not crit1 and n_met <= 1 else
    "PARTIAL")

bar = "=" * 78
print(bar)
print("  假設：Sentinel-2 遙測指數加入 XGBoost 後，房價預測 MAPE 可降低")
print(bar)
print(f"  MAPE_A  {res.MAPE_A.mean():6.2f} ± {res.MAPE_A.std():.2f}")
print(f"  MAPE_B  {res.MAPE_B.mean():6.2f} ± {res.MAPE_B.std():.2f}")
print(f"  ΔMAPE   {main_d:+6.3f} ± {main_sd:.3f} pp"
      f"   {'（大於 seed 雜訊）' if abs(main_d) > main_sd else '（**小於 seed 雜訊**）'}")
print()
print(f"  成功標準 1（ΔMAPE ≥ 2 pp）        ：{'✅' if crit1 else '❌'}")
print(f"  成功標準 2（≥ 3 縣市顯著）        ：{'✅' if crit2 else '❌'}  實得 {n_sig}")
print(f"  成功標準 3（影像進 SHAP 前 10）   ：{'✅' if crit3 else '❌'}")
print()
print(f"  ΔMdAPE  {res['ΔMdAPE'].mean():+6.3f} pp"
      f"   （中位數口徑；MAPE 有 46% 來自最高價的 1/4 cell）")
print(f"  單元 cluster 95% CI [{overall_cl['ci_lo']:+.3f}, {overall_cl['ci_hi']:+.3f}]"
      f"  ICC {overall_cl['icc']:.2f}")
print()
print(f"  N1d 單元輪換（主對照）ΔMAPE {null_d:+.3f} pp")
print(f"  真實影像 vs 輪換影像（同 seed）：{net_d:+.3f} pp"
      f"   ← **影像認不認得出自己的單元**")
print(f"  N2 時間打散         ΔMAPE {shuf_d:+.3f} pp"
      f"   → 屬於時間訊號的 {time_d:+.3f} pp")
if "skipped" not in n3:
    print(f"  N3 季粒度           ΔMAPE {n3['dMAPE_mean']:+.3f} pp"
          f"   {'（符號與主結果一致）' if controls['quarterly_same_sign'] else '（**符號相反**）'}")
print(f"  三層特徵符號一致：{controls['tier_sign_consistent']}")
print(f"  負對照狀態：{controls_status}"
      + ("  （主結果是 null，負對照不適用）" if controls_status == "na_null_result" else ""))
print()
print(f"  達成的成功標準：{n_met} / 3")
print(f"  結論：**{verdict['overall']}**")
print("  註：SUPPORTED=三條全過且負對照通過｜INVALID=三條全過但負對照失敗"
      "｜NOT_SUPPORTED_NEGATIVE=影像顯著讓模型變差｜NOT_SUPPORTED=沒有增量"
      "｜PARTIAL=有正增量但未達標")
if _sig_negative:
    print()
    print("  ⚠️  單元 cluster 的 95% CI 完全落在 0 以下——**影像特徵顯著讓模型變差**，")
    print("      不只是「沒有幫助」。成功標準 3（SHAP 前 10）達成只代表樹用了那個特徵，")
    print("      不代表 out-of-sample 有增量。")
print(bar)

(PROC / "modeling_result.json").write_text(
    json.dumps({"generated_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
                "scheme": MM["scheme"], "n_seeds": N_SEEDS,
                "split": {"train_start": str(TRAIN_START), "train_end": str(TRAIN_END),
                          "val_end": str(VAL_END), "test_end": str(TEST_END)},
                "n_test_rows": int((data.split == "test").sum()),
                "features_A": FEATS_A, "features_img": IMG_FEATS,
                "mape": {"A_mean": round(float(res.MAPE_A.mean()), 3),
                         "A_sd": round(float(res.MAPE_A.std()), 3),
                         "B_mean": round(float(res.MAPE_B.mean()), 3),
                         "B_sd": round(float(res.MAPE_B.std()), 3)},
                "by_city": by_city.round(3).to_dict("index"),
                "city_test": city_test.round(5).to_dict("index"),
                "overall_paired_test_quarter": {
                    k: (None if pd.isna(v) else v) for k, v in overall_q.items()},
                "verdict": verdict,
                "delta_mape_by_seed": res["ΔMAPE"].round(4).tolist(),
                "cluster_test": {k: (None if pd.isna(v) else v)
                                 for k, v in overall_cl.items()},
                "inherited_biases": MM["known_biases"]},
               ensure_ascii=False, indent=1), encoding="utf-8")
print(f"已寫出 {PROC / 'modeling_result.json'}")